In [1]:
from pathlib import Path
import json

from support.data import load
from support.analysis import analyze_product

reviews, products = load(Path("datasets"))

print("Reviews loaded:", len(reviews))
print("Products loaded:", len(products))

Reviews loaded: 6992
Products loaded: 600


In [2]:

product_id = "B0000BYDKO"

result = analyze_product(product_id, reviews)

print("Product:", result["product_id"])
print("Review count:", result["review_count"])
print("Rating counts:", result["rating_counts"])
print("Rating proportions:", result["rating_proportions"])
print("Mean:", result["mean"])
print("Median:", result["median"])


Product: B0000BYDKO
Review count: 30
Rating counts: {1: 5, 2: 4, 3: 8, 4: 5, 5: 8}
Rating proportions: {1: 0.16666666666666666, 2: 0.13333333333333333, 3: 0.26666666666666666, 4: 0.16666666666666666, 5: 0.26666666666666666}
Mean: 3.2333333333333334
Median: 3.0


In [3]:

import numpy as np

# Verificaciones del producto conocido
assert sum(result["rating_counts"].values()) == result["review_count"]
assert np.isclose(sum(result["rating_proportions"].values()), 1.0)

ratings = [
    review["rating"]
    for review in reviews
    if review["product_id"] == product_id
]

assert np.isclose(result["mean"], np.mean(ratings))
assert np.isclose(result["median"], np.median(ratings))

# Producto desconocido
unknown = analyze_product("UNKNOWN_PRODUCT", reviews)

assert unknown["review_count"] == 0
assert all(count == 0 for count in unknown["rating_counts"].values())
assert all(value == 0.0 for value in unknown["rating_proportions"].values())
assert unknown["mean"] is None
assert unknown["median"] is None

# Identificador vacío
for invalid_id in ("", "   "):
    try:
        analyze_product(invalid_id, reviews)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for blank product ID")

# Serialización JSON
json.dumps(result)
json.dumps(unknown)

print("All analyze_product checks passed!")


All analyze_product checks passed!


In [4]:
import importlib
import support.analysis

importlib.reload(support.analysis)

from support.analysis import analyze_product, compare_products

In [5]:

from support.analysis import compare_products

product_ids = [
    "B0000BYDKO",
    "B07CQMZVZ6",
    "B091K4WYD1",
]

comparison = compare_products(product_ids, reviews)

for product, difference in zip(
    comparison["products"],
    comparison["mean_differences"],
):
    print(
        f'{product["product_id"]} | '
        f'Reviews: {product["review_count"]} | '
        f'Mean: {product["mean"]} | '
        f'Difference: {difference}'
    )


B0000BYDKO | Reviews: 30 | Mean: 3.2333333333333334 | Difference: 0.0
B07CQMZVZ6 | Reviews: 30 | Mean: 4.5 | Difference: 1.2666666666666666
B091K4WYD1 | Reviews: 30 | Mean: 4.1 | Difference: 0.8666666666666663


In [6]:

# Comparación válida
assert len(comparison["products"]) == 3
assert len(comparison["mean_differences"]) == 3

assert [
    product["product_id"]
    for product in comparison["products"]
] == product_ids

assert comparison["mean_differences"][0] == 0.0

for product, difference in zip(
    comparison["products"],
    comparison["mean_differences"],
):
    expected = product["mean"] - comparison["products"][0]["mean"]
    assert np.isclose(difference, expected)

# Producto desconocido
unknown_comparison = compare_products(
    ["B0000BYDKO", "UNKNOWN_PRODUCT"],
    reviews,
)

assert unknown_comparison["products"][1]["review_count"] == 0
assert unknown_comparison["mean_differences"][1] is None

# Referencia desconocida
unknown_reference = compare_products(
    ["UNKNOWN_PRODUCT", "B0000BYDKO"],
    reviews,
)

assert all(
    difference is None
    for difference in unknown_reference["mean_differences"]
)

# Entradas inválidas
invalid_cases = [
    [],
    ["B0000BYDKO"],
    ["B0000BYDKO", "B0000BYDKO"],
    ["B0000BYDKO", ""],
    ["B0000BYDKO", "   "],
]

for case in invalid_cases:
    try:
        compare_products(case, reviews)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected ValueError for {case}")

# Serialización
json.dumps(comparison)
json.dumps(unknown_comparison)
json.dumps(unknown_reference)

print("All compare_products checks passed!")
    

All compare_products checks passed!
